In [1]:
import sys
sys.path.insert(1, '../')

In [2]:
import joblib
import pickle

from sklearn.model_selection import StratifiedKFold

from src.data import load_dataset
from src.cv_undersamplers import undersample_data

from src.undersamplers import rus

In [3]:
X_train, X_test, y_train, y_test = load_dataset("abalone_19")

In [4]:
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=10)

X_train = X_train.to_numpy()

for fold_train_idx, fold_test_idx in skf.split(X_train, y_train):
    rus.fit_resample(X_train[fold_train_idx], y_train[fold_train_idx])
    print(fold_train_idx[rus.sample_indices_])
    print()

[1479 1731  737 2292 2562  384 2698 2129  372 2084 1721  117 2225 2611
   26  775  856 1067 1241 1337 1419 1792 1901 1938 2176 2314 2384 2854]

[2479 2177 2145 2523  710 2226 2443 2529 2660 2049  590 2711 1261  677
   26  332  617  775  889 1062 1124 1241 1242 1419 1901 1979 2176 2854]

[2459 2169 2136 2506  675 2217 2425 2512 2654 2045  548 2702 1215  642
  332  617  856  889 1062 1067 1124 1242 1337 1792 1938 1979 2314 2384]



In [5]:
def cv_undersample_split(undersampler, x, y):
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=10)
    folds = []
    for fold_train_idx, fold_test_idx in skf.split(x, y):
        undersampler.fit_resample(x[fold_train_idx], y[fold_train_idx])
        fold_train_sampled_idx = fold_train_idx[undersampler.sample_indices_]
        folds.append((fold_train_sampled_idx, fold_test_idx))
    return folds

folds = cv_undersample_split(rus, X_train, y_train)
folds

[(array([1479, 1731,  737, 2292, 2562,  384, 2698, 2129,  372, 2084, 1721,
          117, 2225, 2611,   26,  775,  856, 1067, 1241, 1337, 1419, 1792,
         1901, 1938, 2176, 2314, 2384, 2854]),
  array([   1,    4,    5,    6,   10,   11,   14,   23,   27,   34,   39,
           40,   41,   44,   47,   53,   54,   55,   56,   58,   59,   63,
           69,   71,   72,   77,   78,   80,   81,   83,   85,   91,   92,
           93,   95,   97,   99,  103,  107,  108,  118,  119,  121,  124,
          126,  146,  147,  149,  150,  154,  157,  158,  159,  166,  167,
          169,  172,  175,  176,  177,  183,  185,  202,  206,  211,  212,
          216,  231,  235,  238,  244,  246,  247,  248,  250,  251,  252,
          253,  259,  260,  263,  266,  268,  271,  272,  274,  276,  282,
          286,  287,  288,  290,  292,  293,  295,  299,  302,  308,  312,
          313,  315,  324,  328,  332,  333,  335,  337,  339,  340,  341,
          345,  349,  353,  358,  360,  366,  368,  3

In [9]:
from src.classical_models import estimator_dict
from src.hyperparams import hyperparam_classical_dict

from sklearn.model_selection import HalvingRandomSearchCV

In [11]:
search = HalvingRandomSearchCV(
      estimator=estimator_dict["rf"],
      param_distributions=hyperparam_classical_dict["rf_params"],
      n_candidates="exhaust",
      factor=3,  # only a third of the candidates are promoted
      resource='n_estimators',  # the limiting resource
      max_resources=500, # max number of trees
      min_resources=10,
      scoring='roc_auc',
      cv=folds,  # <---- use pre-computed folds here
      random_state=10,
      refit=True,
      n_jobs=-1,
      verbose=True
  )
search.fit(X_train, y_train)

n_iterations: 4
n_required_iterations: 4
n_possible_iterations: 4
min_resources_: 10
max_resources_: 500
aggressive_elimination: False
factor: 3
----------
iter: 0
n_candidates: 50
n_resources: 10
Fitting 3 folds for each of 50 candidates, totalling 150 fits
----------
iter: 1
n_candidates: 17
n_resources: 30
Fitting 3 folds for each of 17 candidates, totalling 51 fits
----------
iter: 2
n_candidates: 6
n_resources: 90
Fitting 3 folds for each of 6 candidates, totalling 18 fits
----------
iter: 3
n_candidates: 2
n_resources: 270
Fitting 3 folds for each of 2 candidates, totalling 6 fits


HalvingRandomSearchCV(cv=[(array([1479, 1731,  737, 2292, 2562,  384, 2698, 2129,  372, 2084, 1721,
        117, 2225, 2611,   26,  775,  856, 1067, 1241, 1337, 1419, 1792,
       1901, 1938, 2176, 2314, 2384, 2854]),
                           array([   1,    4,    5,    6,   10,   11,   14,   23,   27,   34,   39,
         40,   41,   44,   47,   53,   54,   55,   56,   58,   59,   63,
         69,   71,   72,   77,   78,   80,   81,   83,   85,   91,   92,
         93,   95,   97,   99,  103,  107,  108,  118,  119,  121,  124,
        126,  146,  147,  149,  150,  154,  157,  158,  159,  166,...
                      max_resources=500, min_resources=10, n_jobs=-1,
                      param_distributions={'max_depth': (1, 2, 3, 4, 8, None),
                                           'max_features': ('log2', 0.25,
                                                            'sqrt', 1.0),
                                           'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x12ae44e90>,
                                           'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x126da2d10>},
                      random_state=10, resource='n_estimators',
                      scoring='roc_auc', verbose=True)